# Review Classification: Structured Outputs vs Decision Models

Classify customer reviews with an LLM using OpenAI structured outputs, then with a decision model (Jev) through OpenRouter, and compare the speed, cost and confidence of each approach.

In [1]:
%pip install openai pandas requests --upgrade --quiet

In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

## The reviews

Each review has a human label, so we can check how often each approach agrees with it. A few are deliberately awkward: sarcasm, mixed feelings and faint praise.

In [3]:
reviews = [
    ("This is a great movie. I will watch it again.", "positive"),
    ("I love this movie!", "positive"),
    ("I hate this movie.", "negative"),
    ("That was a waste of my time.", "negative"),
    ("I will never get that time back.", "negative"),
    ("I will never watch a movie by that director again.", "negative"),
    ("Absolutely fantastic! A must-watch.", "positive"),
    ("The storyline was captivating from start to finish.", "positive"),
    ("The acting was subpar and the plot was predictable.", "negative"),
    ("I was on the edge of my seat the whole time.", "positive"),
    ("The cinematography was breathtaking.", "positive"),
    ("I wouldn't recommend this movie to anyone.", "negative"),
    ("The characters lacked depth and the dialogue was cheesy.", "negative"),
    ("A rollercoaster of emotions. Loved every minute of it.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("The hype around this movie was undeserved.", "negative"),
    ("A refreshing take on a classic story.", "positive"),
    ("Oh great, another two hours of my life I'll never see again.", "negative"),
    ("The first half dragged, but the ending made it all worth it.", "positive"),
    ("Gorgeous to look at, shame about the script.", "negative"),
    ("It was fine. Nothing special, but I didn't hate it.", "positive"),
    ("My kids loved it, I spent most of it checking my phone.", "negative"),
    ("Not the director's best, but still better than most films this year.", "positive"),
    ("I laughed, I cried, I bought the soundtrack.", "positive"),
]

print(f"{len(reviews)} reviews")

24 reviews


## 1. An LLM with structured outputs

Structured outputs constrain the model's generated text to a JSON schema, so the answer always parses into our Pydantic class. The `Literal` type limits the sentiment to two labels.

`gpt-5.6-luna` is the small, fast tier of the GPT-5.6 family. We turn reasoning off because a two-label classification does not need it.

In [4]:
import time
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel

LLM_MODEL = "gpt-5.6-luna"

client = OpenAI()


class ReviewClassification(BaseModel):
    sentiment: Literal["positive", "negative"]


INSTRUCTIONS = "Classify the sentiment of this movie review as positive or negative."


def classify_with_llm(review: str, llm_client=client, model=LLM_MODEL):
    start = time.perf_counter()
    response = llm_client.responses.parse(
        model=model,
        instructions=INSTRUCTIONS,
        input=review,
        text_format=ReviewClassification,
        reasoning={"effort": "none"},
    )
    seconds = time.perf_counter() - start
    return response.output_parsed.sentiment, seconds, response.usage


label, seconds, usage = classify_with_llm(reviews[0][0])
print(label, f"{seconds:.2f}s", usage.input_tokens, "input tokens,", usage.output_tokens, "output tokens")

positive 1.64s 75 input tokens, 14 output tokens


## 2. The same model through OpenRouter: change the base URL

OpenRouter speaks the same API as OpenAI. Point the OpenAI SDK at OpenRouter's base URL, use an OpenRouter key and prefix the model with its provider. Everything else stays the same.

OpenRouter also returns the exact cost of each call in `usage.cost`, which makes the comparison below easy. This is the same trick we use for local models in LM Studio: only the base URL and key change.

In [5]:
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

OPENROUTER_LLM_MODEL = "openai/gpt-5.6-luna"

llm_rows = []
for review, human_label in reviews:
    label, seconds, usage = classify_with_llm(review, openrouter, OPENROUTER_LLM_MODEL)
    llm_rows.append({
        "review": review,
        "human": human_label,
        "llm": label,
        "llm_seconds": seconds,
        "llm_cost": usage.cost,
    })
    print(f"{seconds:5.2f}s  {label:8}  {review}")

 1.37s  positive  This is a great movie. I will watch it again.


 1.36s  positive  I love this movie!


 1.24s  negative  I hate this movie.


 1.10s  negative  That was a waste of my time.


 1.09s  negative  I will never get that time back.


...

 0.99s  negative  Gorgeous to look at, shame about the script.


 1.00s  positive  It was fine. Nothing special, but I didn't hate it.


 0.93s  negative  My kids loved it, I spent most of it checking my phone.


 2.22s  positive  Not the director's best, but still better than most films this year.


 1.07s  positive  I laughed, I cried, I bought the soundtrack.


## 3. A decision model: Jev through OpenRouter

Jev, from TypeSafe, is a *System One* decision model. It does not generate text. You send it some **state** (the review) and typed **questions**, and it returns typed answers with **calibrated probabilities**:

| Question type | Returns |
|---|---|
| `choice` | one option from your list, plus a probability for each |
| `noul` | the probability that a yes/no statement is true |
| `score` | a position on an ordered scale you define |

There is no JSON to prompt for and nothing to parse. OpenRouter serves it through its Decisions API, which we call with `requests`.

In [6]:
import requests

DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"
DECISION_MODEL = "~typesafe/jev-latest"

HEADERS = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"}


def decide(state, questions):
    start = time.perf_counter()
    response = requests.post(
        DECISIONS_URL,
        headers=HEADERS,
        json={"model": DECISION_MODEL, "state": state, "questions": questions},
        timeout=60,
    )
    response.raise_for_status()
    return response.json(), time.perf_counter() - start


SENTIMENT_QUESTION = {
    "sentiment": {
        "type": "choice",
        "instructions": "Classify the sentiment of this movie review.",
        "criteria": {
            "positive": "The reviewer liked the movie overall",
            "negative": "The reviewer disliked the movie overall",
        },
    }
}

result, seconds = decide(reviews[0][0], SENTIMENT_QUESTION)
print(f"{seconds:.2f}s")
result["answers"], result["usage"]

0.44s


({'sentiment': {'type': 'choice',
   'choice': 'positive',
   'probabilities': {'negative': 0, 'positive': 1},
   'confidence': 1}},
 {'input_tokens': 332, 'output_tokens': 32, 'cost': 1.3944e-05})

In [7]:
for row in llm_rows:
    result, seconds = decide(row["review"], SENTIMENT_QUESTION)
    answer = result["answers"]["sentiment"]
    row.update({
        "jev": answer["choice"],
        "jev_confidence": answer["confidence"],
        "jev_seconds": seconds,
        "jev_cost": result["usage"]["cost"],
    })
    print(f"{seconds:5.2f}s  {answer['choice']:8}  {answer['confidence']:.2f}  {row['review']}")

 0.31s  positive  1.00  This is a great movie. I will watch it again.


 0.30s  positive  1.00  I love this movie!


 0.47s  negative  1.00  I hate this movie.


 0.28s  negative  1.00  That was a waste of my time.


 0.33s  negative  0.99  I will never get that time back.


...

 0.37s  negative  0.98  Gorgeous to look at, shame about the script.


 0.30s  positive  0.58  It was fine. Nothing special, but I didn't hate it.


 0.27s  negative  0.54  My kids loved it, I spent most of it checking my phone.


 0.44s  positive  0.98  Not the director's best, but still better than most films this year.


 1.24s  positive  1.00  I laughed, I cried, I bought the soundtrack.


## 4. Compare speed, cost and agreement

Both runs went through OpenRouter, so latency and cost are measured the same way. Your numbers will vary with network, load and pricing, but the shape of the difference is what matters: the LLM pays for every generated token and waits for them, while the decision model only reads.

In [8]:
import pandas as pd

df = pd.DataFrame(llm_rows)

summary = pd.DataFrame({
    "LLM + structured outputs": {
        "median seconds per review": df["llm_seconds"].median(),
        "total seconds": df["llm_seconds"].sum(),
        "total cost ($)": df["llm_cost"].sum(),
        "cost per 1,000 reviews ($)": df["llm_cost"].mean() * 1000,
        "agrees with human label": (df["llm"] == df["human"]).mean(),
    },
    "Decision model (Jev)": {
        "median seconds per review": df["jev_seconds"].median(),
        "total seconds": df["jev_seconds"].sum(),
        "total cost ($)": df["jev_cost"].sum(),
        "cost per 1,000 reviews ($)": df["jev_cost"].mean() * 1000,
        "agrees with human label": (df["jev"] == df["human"]).mean(),
    },
})

summary.round(6)

,LLM + structured outputs,Decision model (Jev)
median seconds per review,1.049953,0.351422
total seconds,26.927529,10.952446
total cost ($),0.000746,0.000333
"cost per 1,000 reviews ($)",0.031086,0.013876
agrees with human label,1.000000,1.000000


In [9]:
speedup = df["llm_seconds"].median() / df["jev_seconds"].median()
cost_ratio = df["llm_cost"].sum() / df["jev_cost"].sum()
print(f"Decision model: {speedup:.1f}x faster per review (median), {cost_ratio:.1f}x cheaper overall")

# Where the two approaches disagree:
df.loc[df["llm"] != df["jev"], ["review", "human", "llm", "jev", "jev_confidence"]]

Decision model: 3.0x faster per review (median), 2.2x cheaper overall


,review,human,llm,jev,jev_confidence


## 5. Calibrated confidence: decide what to automate

An LLM with structured outputs gives you a label and nothing else. Jev gives a probability that is **calibrated**: answers at 0.9 confidence are right about 90% of the time. That lets you set a threshold, handle confident answers automatically and send the rest to a person.

In [10]:
THRESHOLD = 0.9

df["route"] = df["jev_confidence"].apply(lambda c: "automatic" if c >= THRESHOLD else "human review")
print(df["route"].value_counts().to_string(), "\n")

df.sort_values("jev_confidence")[["review", "jev", "jev_confidence", "route"]].head(8)

route
automatic       22
human review     2 



,review,jev,jev_confidence,route
21,"My kids loved it, I spent most of it checking ...",negative,0.54,human review
20,"It was fine. Nothing special, but I didn't hat...",positive,0.58,human review
19,"Gorgeous to look at, shame about the script.",negative,0.98,automatic
22,"Not the director's best, but still better than...",positive,0.98,automatic
4,I will never get that time back.,negative,0.99,automatic
9,I was on the edge of my seat the whole time.,positive,0.99,automatic
0,This is a great movie. I will watch it again.,positive,1.00,automatic
18,"The first half dragged, but the ending made it...",positive,1.00,automatic


## 6. Several typed questions in one call

One decision call can answer several questions about the same state at once. Here we triage a product review: the sentiment, whether it needs a reply, and how severe the problem is.

In [11]:
support_review = "The handle snapped off on day two and nobody has answered my emails."

triage_questions = {
    "sentiment": {
        "type": "choice",
        "instructions": "What is the overall sentiment of this review?",
        "criteria": {
            "positive": "The customer is happy",
            "neutral": "Mixed, or no clear feeling",
            "negative": "The customer is unhappy",
        },
    },
    "needs_reply": {
        "type": "noul",
        "instructions": "The customer needs a reply from the support team.",
    },
    "severity": {
        "type": "score",
        "instructions": "How severe is the customer's problem?",
        "criteria": ["No problem", "Minor", "Major", "Product unusable"],
    },
}

result, seconds = decide(support_review, triage_questions)
answers = result["answers"]

print(f"{seconds:.2f}s, ${result['usage']['cost']:.6f}")
print("Sentiment:  ", answers["sentiment"]["choice"], answers["sentiment"]["confidence"])
print("Needs reply:", answers["needs_reply"]["noul"])
print("Severity:   ", answers["severity"]["score"], "on a 0-3 scale")

0.43s, $0.000017
Sentiment:   negative 1
Needs reply: 0.95
Severity:    2.97 on a 0-3 scale


## Summary

- **Structured outputs** make an LLM's answer parse reliably into a schema, but the model still generates every token, so you pay for output and wait for it.
- **Changing the base URL** lets the same OpenAI SDK call OpenRouter (or a local server) with no other code changes.
- **Decision models** such as Jev return typed answers without generating text: faster and cheaper for fixed label sets, with several questions answered in one call.
- **Calibrated probabilities** turn confidence into a routing rule: automate the confident answers and send the uncertain ones to a person.
- Use an LLM when you need free text, extraction or reasoning in the answer; use a decision model for high-volume classification, routing and moderation.